# Water Potability — Part 4: Hyperparameter Tuning and Optimization

**Student:** Joel Elizee  
**Student ID:** joeeli7267  

Three models from Part 3: Random Forest classification, Random Forest pH regression, and K-means clustering. Random search tunes both forests; a small exhaustive grid tunes K-means. **Held-out test data is never used to choose hyperparameters.**

## 1. Data, consistent split, and leakage-safe preprocessing

Repeat Part 3’s 80/20 stratified split (seed 42). Fit imputation, scaling, and PCA on training only, through pipelines. The regression target is observed pH; rows missing that **target** are excluded, while missing **predictor** values are imputed. `Potability` is excluded from clustering.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             mean_absolute_error, mean_squared_error, r2_score, silhouette_score)
SEED=42
df=pd.read_csv('water_potability.csv')
features=df.drop(columns='Potability').columns.tolist()
itr,ite=train_test_split(df.index,test_size=.20,random_state=SEED,stratify=df.Potability)
train_df,test_df=df.loc[itr],df.loc[ite]
X_train,X_test=train_df[features],test_df[features]
y_train,y_test=train_df.Potability,test_df.Potability
print('Train/test:',len(train_df),len(test_df))
print('Missing values:',df.isna().sum().to_dict())

## 2. Classification — Random Forest

**Baseline:** 120 trees, depth 8, minimum leaf 4, balanced class weights. **Tuning:** 6 random combinations, 3-fold stratified CV, optimize *F1 for potable water* on training data. Depth and minimum leaf size control tree complexity (regularization). F1 is emphasized because potable-water false negatives and false positives should not be obscured by accuracy alone.

In [ ]:
baseline_classifier=Pipeline([('imputer',SimpleImputer(strategy='median')),
 ('forest',RandomForestClassifier(n_estimators=120,max_depth=8,min_samples_leaf=4,class_weight='balanced',random_state=42,n_jobs=1))])
baseline_classifier.fit(X_train,y_train)
classifier_search=RandomizedSearchCV(Pipeline([('imputer',SimpleImputer(strategy='median')),
 ('forest',RandomForestClassifier(class_weight='balanced',random_state=42,n_jobs=1))]),
 param_distributions={'forest__n_estimators':[40,80], 'forest__max_depth':[4,6,8,12],
 'forest__min_samples_leaf':[3,6,10], 'forest__max_features':['sqrt',0.7]},
 n_iter=6,scoring='f1',cv=StratifiedKFold(3,shuffle=True,random_state=42),
 n_jobs=1,random_state=42,refit=True)
classifier_search.fit(X_train,y_train)
optimized_classifier=classifier_search.best_estimator_
def classifier_scores(model):
 pred=model.predict(X_test)
 return {'accuracy':accuracy_score(y_test,pred),'precision':precision_score(y_test,pred,zero_division=0),
 'recall':recall_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0),
 'train_accuracy':accuracy_score(y_train,model.predict(X_train))}
print('CV best:',classifier_search.best_score_,'parameters:',classifier_search.best_params_)
print('Baseline:',classifier_scores(baseline_classifier))
print('Optimized:',classifier_scores(optimized_classifier))
print('Optimized test confusion matrix:',confusion_matrix(y_test,optimized_classifier.predict(X_test)))

### Classification evaluation (held-out test)

Baseline accuracy **0.662**, F1 **0.502**; optimized accuracy **0.637**, F1 **0.480**. Train/test gaps can signal overfitting; test accuracy is not the search objective. Tree depth and minimum leaf size serve as complexity regularization. Random Forest has no neural-network epoch loss history; CV F1 and prediction errors are the relevant metrics.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig,axes=plt.subplots(1,2,figsize=(9,3.5))
for ax,model,label in zip(axes,[baseline_classifier,optimized_classifier],['Baseline','Optimized']):
 ConfusionMatrixDisplay.from_estimator(model,X_test,y_test,ax=ax,colorbar=False,cmap='Blues')
 ax.set_title(label)
plt.tight_layout();plt.show()

## 3. Regression — Random Forest to estimate pH

**Baseline:** 120 trees, maximum depth 9, minimum leaf 4. **Tuning:** 6 random combinations, 3-fold CV scored by negative MAE (lower error is better). Minimum leaf size and capped depth regularize the forest. Do not impute missing pH target labels; exclude them. Compare R² and MAE with the Part 3 model.

In [ ]:
reg_predictors=[c for c in features if c!='ph']
ph_train=train_df.dropna(subset=['ph']);ph_test=test_df.dropna(subset=['ph'])
base_regressor=Pipeline([('imputer',SimpleImputer(strategy='median')),
 ('forest',RandomForestRegressor(n_estimators=120,max_depth=9,min_samples_leaf=4,random_state=42,n_jobs=1))])
base_regressor.fit(ph_train[reg_predictors],ph_train.ph)
regressor_search=RandomizedSearchCV(Pipeline([('imputer',SimpleImputer(strategy='median')),
 ('forest',RandomForestRegressor(random_state=42,n_jobs=1))]),
 param_distributions={'forest__n_estimators':[40,80], 'forest__max_depth':[3,5,8,12],
 'forest__min_samples_leaf':[5,10,20], 'forest__max_features':['sqrt',0.7,1.0]},
 n_iter=6,scoring='neg_mean_absolute_error',cv=KFold(3,shuffle=True,random_state=42),
 n_jobs=1,random_state=42,refit=True)
regressor_search.fit(ph_train[reg_predictors],ph_train.ph)
optimized_regressor=regressor_search.best_estimator_
def regressor_scores(model):
 pred=model.predict(ph_test[reg_predictors]);true=ph_test.ph
 return {'MAE':mean_absolute_error(true,pred),'RMSE':np.sqrt(mean_squared_error(true,pred)),
 'R2':r2_score(true,pred),'train_R2':r2_score(ph_train.ph,model.predict(ph_train[reg_predictors]))}
print('Best CV MAE:',-regressor_search.best_score_,'parameters:',regressor_search.best_params_)
print('Baseline:',regressor_scores(base_regressor))
print('Optimized:',regressor_scores(optimized_regressor))

### Regression evaluation (held-out test)

MAE changed from **1.225** to **1.225**; R² changed from **0.003** to **0.013**. MAE/RMSE are error or loss measures; lower is better. R² close to zero means chemistry predictors explain little variation in held-out pH. Hyperparameter search does not create predictive signal.

In [ ]:
fig,ax=plt.subplots(figsize=(6,4))
pred=optimized_regressor.predict(ph_test[reg_predictors])
ax.scatter(ph_test.ph,pred,alpha=.5,s=15);ax.plot([2,13],[2,13],'k--')
ax.set(xlabel='Observed pH',ylabel='Predicted pH',title='Optimized model: predicted vs. observed pH')
plt.tight_layout();plt.show()

## 4. Clustering — K-means

**Baseline:** K-means with k=3, n_init=10 on median-imputed, standardized measurements. **Tuning:** exhaustive search over k=2…6 using training silhouette, with `n_init=10`. **Regularization:** training-only PCA retaining 90% of variance reduces noisy/redundant dimensions before K-means; this is *dimensionality-based regularization*, **not** an L2 penalty or neural-network dropout. There is no supervised target or classification accuracy for K-means. Report train/test silhouette and squared-distance inertia. Because PCA changes representation, inertia is **not directly comparable** between baseline and optimized models.

In [ ]:
baseline_prep=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
Zbase_train=baseline_prep.fit_transform(X_train);Zbase_test=baseline_prep.transform(X_test)
baseline_clusters=KMeans(n_clusters=3,n_init=10,random_state=42).fit(Zbase_train)
cluster_prep=Pipeline([('imputer',SimpleImputer(strategy='median')),
 ('scaler',StandardScaler()),('pca',PCA(n_components=0.90,svd_solver='full'))])
Z_train=cluster_prep.fit_transform(X_train);Z_test=cluster_prep.transform(X_test)
cluster_search=[]
for k in [2,3,4,5,6]:
 candidate=KMeans(n_clusters=k,n_init=10,random_state=42).fit(Z_train)
 score=silhouette_score(Z_train,candidate.labels_,sample_size=min(1200,len(Z_train)),random_state=42)
 cluster_search.append((k,score))
best_k=max(cluster_search,key=lambda item:item[1])[0]
optimized_clusters=KMeans(n_clusters=best_k,n_init=10,random_state=42).fit(Z_train)
def clustering_scores(model,train,test):
 lab=model.predict(test)
 return {'train_silhouette':silhouette_score(train,model.labels_,sample_size=min(1200,len(train)),random_state=42),
 'test_silhouette':silhouette_score(test,lab), 'train_inertia_per_sample':model.inertia_/len(train),
 'test_inertia_per_sample':np.sum((test-model.cluster_centers_[lab])**2)/len(test)}
print('Search k/silhouette:',cluster_search,'selected k:',best_k,'PCA features:',Z_train.shape[1])
print('Baseline:',clustering_scores(baseline_clusters,Zbase_train,Zbase_test))
print('Optimized:',clustering_scores(optimized_clusters,Z_train,Z_test))

### Clustering evaluation (held-out test)

Test silhouette changed from **0.072** to **0.082** after PCA and tuning k to **2**. Silhouette is a quality score, not classification accuracy. K-means optimizes inertia during fitting; there is no epoch loss curve. Cluster labels are not safety grades, and tests should examine whether any profiles are scientifically interpretable.

In [ ]:
fig,ax=plt.subplots(figsize=(6,3.5));ax.plot(*zip(*cluster_search),marker='o');ax.set(xlabel='k',ylabel='Train silhouette',title='Tuning K-means cluster count');plt.tight_layout();plt.show()
view=PCA(n_components=2).fit_transform(Z_train)
fig,ax=plt.subplots(figsize=(6,4));ax.scatter(view[:,0],view[:,1],c=optimized_clusters.labels_,s=10,cmap='tab10',alpha=.6)
ax.set(title='Optimized K-means: 2D view',xlabel='PC1',ylabel='PC2');plt.tight_layout();plt.show()

## 5. Summary and limitations

Compare all metrics against Part 3, including any declines. Model selection used training-fold CV only, followed by one held-out evaluation; repeated testing can still lead to optimistic results if it guides future changes. Model complexity is restrained using forest depth/leaf limits and PCA. No algorithm here is a neural network, so neural-network activation functions, dropout and compile-time epoch loss are inapplicable. The available chemical measurements should not be used to certify drinking-water safety without independent validation.